In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string

# As your ML teacher, let's understand these crucial parameters for date conversion:

# When we use `pd.to_datetime(df['date'], format='mixed', errors='coerce')`:

# 1. `format='mixed'`:
#    - **What it does:** This tells pandas to try and infer the date and time format for *each individual string* in the 'date' column.
#    - **Why it's useful in ML:** Real-world data often comes from different sources or is entered inconsistently. You might have 'YYYY-MM-DD', 'MM/DD/YYYY', 'Month Day, Year', or 'DD-Mon-YY' all within the same column. Instead of you having to figure out every single format and write complex parsing logic, `format='mixed'` attempts to intelligently guess the correct format for each entry.
#    - **Think of it as:** A flexible detective that can recognize many different date patterns without explicit instructions for each one.

# 2. `errors='coerce'`:
#    - **What it does:** If pandas encounters a date string that it *cannot* parse (even with `format='mixed'`) into a valid datetime object, instead of raising an error and stopping the entire process, it will replace that unparseable entry with `NaT` (Not a Time).
#    - **Why it's useful in ML:** Data quality is rarely perfect. `errors='coerce'` allows your data pipeline to continue running even if there are a few bad date entries. These `NaT` values can then be handled using standard missing data techniques (e.g., dropping the rows, imputing, etc.), which is a common step in ML preprocessing.
#    - **Think of it as:** A robust error handler that gracefully manages malformed data, preventing your script from crashing and allowing you to deal with invalid data systematically later.

# **In summary, for ML:**
# Using `format='mixed'` and `errors='coerce'` together makes your date conversion highly resilient to diverse and imperfect date formats, which is a common challenge in data cleaning and feature engineering for machine learning models.

# Example of how we used it (from your notebook):
# df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')

In [ ]:
df_fake = pd.read_csv('/content/Fake.csv', encoding='latin', engine='python', on_bad_lines='skip')
df_true = pd.read_csv('/content/True.csv', encoding='latin', engine='python', on_bad_lines='skip')
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Yearâ...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obamaâs Na...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
df_fake.tail()

,title,text,subject,date
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied âSafe Zoneâ Plan ...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016"


In [ ]:
df_fake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [ ]:
## firsty we have to convert date column in date type
df_fake['date'] = pd.to_datetime(df_fake['date'], format='mixed', errors='coerce')
df_true['date'] = pd.to_datetime(df_true['date'], format='mixed', errors='coerce')

In [ ]:
df_fake.isnull().sum()

,0
title,0
text,0
subject,0
date,10


In [ ]:
df_true.isnull().sum()

,0
title,0
text,0
subject,0
date,0


In [ ]:
df_fake.shape


(23481, 4)

In [ ]:
df_true.shape

(21417, 4)

## so we have to remove 10 rows from date column from 23k which is app. 0.04 % of whole data

In [ ]:
blanks_rows=df_fake['text'].isna()
print(f'count of nan text rows:  {blanks_rows.sum()}')

count of nan text rows:  0


## feature Engineering

In [ ]:
# now we create a column "status"
#df_fake[status]='0' and df_true[status] = '1'

In [ ]:
df_fake.insert(loc=4, column='status', value=0)
df_true.insert(loc=4, column='status', value=1)

In [ ]:
df_fake.shape, df_true.shape

((23481, 5), (21417, 5))

In [ ]:
#
df_fake_manualTesting=df_fake.tail(10)
for i in range(10235,10225,-1):
  df_fake.drop([i],axis=0,inplace=True)

df_true_manualTesting=df_true.tail(10)
for i in range (12686,12676,-1):
  df_true.drop([i],axis=0,inplace=True)

In [ ]:
df_fake.shape, df_true.shape

((23471, 5), (21407, 5))

In [ ]:
df_fake_manualTesting.head(10)

,title,text,subject,date,status
23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,2016-01-20,0
23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,2016-01-19,0
23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,2016-01-19,0
23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,2016-01-19,0
23475,Hillary Clinton: âIsrael Firstâ (and no pe...,Robert Fantina CounterpunchAlthough the United...,Middle-east,2016-01-18,0
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,2016-01-16,0
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,2016-01-16,0
23478,Sunnistan: US and Allied âSafe Zoneâ Plan ...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,2016-01-15,0
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,2016-01-14,0
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,2016-01-12,0


In [ ]:
df_true_manualTesting.head(10)

,title,text,subject,date,status
21407,"Mata Pires, owner of embattled Brazil builder ...","SAO PAULO (Reuters) - Cesar Mata Pires, the ow...",worldnews,2017-08-22,1
21408,"U.S., North Korea clash at U.N. forum over nuc...",GENEVA (Reuters) - North Korea and the United ...,worldnews,2017-08-22,1
21409,"U.S., North Korea clash at U.N. arms forum on ...",GENEVA (Reuters) - North Korea and the United ...,worldnews,2017-08-22,1
21410,Headless torso could belong to submarine journ...,COPENHAGEN (Reuters) - Danish police said on T...,worldnews,2017-08-22,1
21411,North Korea shipments to Syria chemical arms a...,UNITED NATIONS (Reuters) - Two North Korean sh...,worldnews,2017-08-21,1
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,2017-08-22,1
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,2017-08-22,1
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,2017-08-22,1
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,2017-08-22,1
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,2017-08-22,1


In [ ]:
df_manual_testing=pd.concat([df_fake_manualTesting,df_true_manualTesting],axis=0)
# df_manual_testing.head()
df_manual_testing.to_csv("manual_testing.csv")

##merge true and fake dataframe

In [ ]:
df_merge=pd.concat([df_fake,df_true],axis=0)
df_merge.head()

,title,text,subject,date,status
0,Donald Trump Sends Out Embarrassing New Yearâ...,Donald Trump just couldn t wish all Americans ...,News,2017-12-31,0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,2017-12-31,0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,2017-12-30,0
3,Trump Is So Obsessed He Even Has Obamaâs Na...,"On Christmas day, Donald Trump announced that ...",News,2017-12-29,0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,2017-12-25,0


In [ ]:
df_merge.columns

Index(['title', 'text', 'subject', 'date', 'status'], dtype='object')

#keep the main column and remove column that not needed

In [ ]:
df_merge.drop(columns=['title','subject','date'],axis=1,inplace=True)

In [ ]:
df_merge.isnull().sum()

,0
text,0
status,0


#Random Sampling of dataframe

In [ ]:
df=df_merge.sample(frac=1)
# df_merge.head()

In [ ]:
df.head()

,text,status
11252,(Reuters) - Millennnials in the West of the Un...,1
14962,PARIS (Reuters) - France s foreign ministry su...,1
17543,When four Chicago thugs videotaped the brutal ...,0
12335,BEIJING/TAIPEI (Reuters) - China accused the U...,1
21612,Not a bad imitation of a black pastor from a g...,0


In [ ]:
# reset index
df.reset_index(inplace=True)
df.drop(['index'],axis=1,inplace=True)

In [ ]:
df.head()

,text,status
0,(Reuters) - Millennnials in the West of the Un...,1
1,PARIS (Reuters) - France s foreign ministry su...,1
2,When four Chicago thugs videotaped the brutal ...,0
3,BEIJING/TAIPEI (Reuters) - China accused the U...,1
4,Not a bad imitation of a black pastor from a g...,0


#creating the function to process the text

In [ ]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1972/2411212894.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/tmp/ipykernel_1972/2411212894.py:5: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
/tmp/ipykernel_1972/2411212894.py:9: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [ ]:
df['text']=df['text'].apply(wordopt)

#dependent and independent variable

In [ ]:
X=df['text']
y=df['status']

#training and testing split

the random =1 state is to reproduce the same split each time you run your program)

In [ ]:
x_train,x_test,ytrain,y_test=train_test_split(X,y,test_size=0.25,random_state=1)

#Convert text into vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vector=TfidfVectorizer()

xv_test=vector.fit_transform(x_test)
xv_train=vector.transform(x_train)

# build logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression

lr=LogisticRegression()

lr.fit(xv_train , ytrain)

LogisticRegression()

In [ ]:
x_predict=lr.predict(xv_test)

In [ ]:
lr.score(xv_test,y_test)

0.9916221033868092

In [ ]:
print(classification_report(x_predict,y_test))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5765
           1       0.99      0.99      0.99      5455

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220



#desicion  tree classification

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt=DecisionTreeClassifier()

dt.fit(xv_train,ytrain)

DecisionTreeClassifier()

In [ ]:
pred_dt=dt.predict(xv_test)

In [ ]:
dt.score(xv_test,y_test)

0.9971479500891266

In [ ]:
print(classification_report(pred_dt,y_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5781
           1       1.00      1.00      1.00      5439

    accuracy                           1.00     11220
   macro avg       1.00      1.00      1.00     11220
weighted avg       1.00      1.00      1.00     11220



#Model Testing


In [ ]:
def output_label(n):
  if n==0:
    return "Fake News"
  elif n==1:
    return "True News"

def Testing(news):
  input_text={"text":[news]}
  df_test=pd.DataFrame(input_text)
  df_test['text']=df_test['text'].apply(wordopt)
  new_df_test=vector.transform(df_test['text'])
  pred_lr=lr.predict(new_df_test)
  pred_dt=dt.predict(new_df_test)

  return print(f'\n LR Prediction : {output_label(pred_lr[0])} \n DT Prediction : {output_label(pred_dt[0])}')

In [ ]:
news= str(input())
Testing(news)

 Vic Bishop Waking TimesOur reality is carefully constructed by powerful corporate, political and special interest sources in order to covertly sway public opinion. Blatant lies are often televised regarding terrorism, food, war, health, etc. They are fashioned to sway public opinion and condition viewers to accept what have become destructive societal norms.The practice of manipulating and controlling public opinion with distorted media messages has become so common that there is a whole industry formed around this. The entire role of this brainwashing industry is to figure out how to spin information to journalists, similar to the lobbying of government. It is never really clear just how much truth the journalists receive because the news industry has become complacent. The messages that it presents are shaped by corporate powers who often spend millions on advertising with the six conglomerates that own 90% of the media:General Electric (GE), News-Corp, Disney, Viacom, Time Warner, 

In [ ]:
news= str(input())
Testing(news)

SAO PAULO (Reuters) - Cesar Mata Pires, the owner and co-founder of Brazilian engineering conglomerate OAS SA, one of the largest companies involved in Brazil s corruption scandal, died on Tuesday. He was 68. Mata Pires died of a heart attack while taking a morning walk in an upscale district of S o Paulo, where OAS is based, a person with direct knowledge of the matter said. Efforts to contact his family were unsuccessful. OAS declined to comment. The son of a wealthy cattle rancher in the northeastern state of Bahia, Mata Pires  links to politicians were central to the expansion of OAS, which became Brazil s No. 4 builder earlier this decade, people familiar with his career told Reuters last year. His big break came when he befriended Antonio Carlos Magalh es, a popular politician who was Bahia governor several times, and eventually married his daughter Tereza. Brazilians joked that OAS stood for  Obras Arranjadas pelo Sogro  - or  Work Arranged by the Father-In-Law.   After years of